In [ ]:
# 필수 라이브러리
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 로딩 완료")

In [ ]:
# 실제 제주 설화 데이터 로딩
data_path = r"d:\repos\tonylee\goorm\oreumi-bull4team\team-data\jeju-stories\scripts\training_data\jeju_folklore_training_data.json"

print(f"📂 실제 데이터 파일 로딩: {os.path.basename(data_path)}")
print("=" * 60)

# JSON 파일 읽기
with open(data_path, 'r', encoding='utf-8') as f:
    jeju_data = json.load(f)

# 메타데이터 확인
metadata = jeju_data['metadata']
samples = jeju_data['training_samples']

print(f"📊 데이터셋 메타데이터:")
print(f"   • 총 샘플 수: {metadata['total_samples']}개")
print(f"   • 원본 스토리: {metadata['source_stories']}개")
print(f"   • 생성 날짜: {metadata['generation_date']}")
print(f"   • 토큰 시스템 버전: {metadata['token_system_version']}")

print(f"\n✅ 실제 샘플 수: {len(samples)}개")

# DataFrame 생성
df = pd.DataFrame(samples)

print(f"\n📋 DataFrame 정보:")
print(f"   • Shape: {df.shape}")
print(f"   • Columns: {list(df.columns)}")
print(f"   • 데이터 타입: {df.dtypes.to_dict()}")

In [ ]:
# 실제 데이터의 문제점 분석
print("🔍 실제 데이터 품질 분석")
print("=" * 40)

# 응답 길이 분석
response_lengths = df['response'].apply(len)
prompt_lengths = df['prompt'].apply(len)

print(f"📊 응답 길이 통계:")
print(f"   • 평균: {response_lengths.mean():.1f} 글자")
print(f"   • 중간값: {response_lengths.median():.1f} 글자")
print(f"   • 최소: {response_lengths.min()} 글자")
print(f"   • 최대: {response_lengths.max()} 글자")
print(f"   • 표준편차: {response_lengths.std():.1f}")

print(f"\n📊 프롬프트 길이 통계:")
print(f"   • 평균: {prompt_lengths.mean():.1f} 글자")
print(f"   • 중간값: {prompt_lengths.median():.1f} 글자")

# 문제점 식별
short_responses = (response_lengths < 100).sum()
very_short_responses = (response_lengths < 50).sum()

print(f"\n⚠️ 문제점 발견:")
print(f"   • 100글자 미만: {short_responses}/{len(df)} ({short_responses/len(df)*100:.1f}%)")
print(f"   • 50글자 미만: {very_short_responses}/{len(df)} ({very_short_responses/len(df)*100:.1f}%)")

if response_lengths.mean() < 200:
    print(f"   ❌ 심각한 문제: 평균 응답이 너무 짧음 (권장: 500+ 글자)")
    print(f"   📝 권장 개선: 응답을 {500/response_lengths.mean():.1f}배 확장 필요")

# 샘플별 분석
print(f"\n📝 가장 짧은 응답 5개:")
shortest = df.loc[response_lengths.nsmallest(5).index]
for idx, (_, row) in enumerate(shortest.iterrows()):
    print(f"   {idx+1}. {row['id']}: {len(row['response'])}글자")
    print(f"      \"{row['response'][:80]}...\"")

print(f"\n📝 가장 긴 응답 5개:")
longest = df.loc[response_lengths.nlargest(5).index]
for idx, (_, row) in enumerate(longest.iterrows()):
    print(f"   {idx+1}. {row['id']}: {len(row['response'])}글자")
    print(f"      \"{row['response'][:80]}...\"")

In [ ]:
# 데이터 구조 및 내용 분석
print("🔍 데이터 구조 분석")
print("=" * 30)

# 스토리 유형별 분석
source_stories = []
target_styles = []

for _, row in df.iterrows():
    metadata = row['metadata']
    if 'source_story' in metadata:
        source_stories.append(metadata['source_story'])
    if 'target_style' in metadata:
        target_styles.append(metadata['target_style'])

if source_stories:
    story_counts = pd.Series(source_stories).value_counts()
    print(f"📚 원본 스토리별 분포:")
    for story, count in story_counts.head(10).items():
        print(f"   • {story}: {count}개")

if target_styles:
    style_counts = pd.Series(target_styles).value_counts()
    print(f"\n🎭 스타일별 분포:")
    for style, count in style_counts.items():
        print(f"   • {style}: {count}개")

# 프롬프트 패턴 분석
print(f"\n🎯 프롬프트 패턴 분석:")
prompt_patterns = defaultdict(int)
for prompt in df['prompt']:
    if '[STYLE:' in prompt:
        style_part = prompt.split('[STYLE:')[1].split(']')[0]
        prompt_patterns[f'STYLE:{style_part}'] += 1
    if '[CHR:' in prompt:
        chr_part = prompt.split('[CHR:')[1].split(']')[0]
        prompt_patterns[f'CHR:{chr_part}'] += 1

print(f"   주요 패턴:")
for pattern, count in sorted(prompt_patterns.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"   • {pattern}: {count}회")

# 문화적 요소 분석
cultural_keywords = ['제주', '서사무가', '본풀이', '굿', '차사', '천지왕', '할망', '신화', '무속', '심방']
cultural_scores = []

for _, row in df.iterrows():
    text = f"{row['prompt']} {row['response']}"
    score = sum(1 for keyword in cultural_keywords if keyword in text)
    cultural_scores.append(score)

print(f"\n🏛️ 문화적 풍부도:")
print(f"   • 평균 문화 키워드: {np.mean(cultural_scores):.1f}개")
print(f"   • 문화 요소 풍부한 샘플: {sum(1 for score in cultural_scores if score >= 3)}/{len(df)}개")

In [ ]:
# 시각화
print("📊 실제 데이터 시각화")
print("=" * 25)

plt.figure(figsize=(15, 10))

# 1. 응답 길이 분포
plt.subplot(2, 3, 1)
plt.hist(response_lengths, bins=20, color='lightcoral', alpha=0.7, edgecolor='black')
plt.title('Response Length Distribution\n(실제 데이터)')
plt.xlabel('Characters')
plt.ylabel('Frequency')
plt.axvline(response_lengths.mean(), color='red', linestyle='--', label=f'Mean: {response_lengths.mean():.0f}')
plt.axvline(200, color='green', linestyle='--', label='Target: 200+')
plt.legend()
plt.grid(True, alpha=0.3)

# 2. 프롬프트 vs 응답 길이
plt.subplot(2, 3, 2)
plt.scatter(prompt_lengths, response_lengths, alpha=0.6, color='skyblue')
plt.title('Prompt vs Response Length')
plt.xlabel('Prompt Length')
plt.ylabel('Response Length')
plt.grid(True, alpha=0.3)

# 3. 스토리별 분포 (상위 10개)
if source_stories:
    plt.subplot(2, 3, 3)
    story_counts.head(10).plot(kind='bar', color='lightgreen', alpha=0.7)
    plt.title('Top 10 Source Stories')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)

# 4. 스타일별 분포
if target_styles:
    plt.subplot(2, 3, 4)
    style_counts.plot(kind='pie', autopct='%1.1f%%')
    plt.title('Style Distribution')

# 5. 문화적 풍부도 분포
plt.subplot(2, 3, 5)
plt.hist(cultural_scores, bins=range(max(cultural_scores)+2), color='orange', alpha=0.7, edgecolor='black')
plt.title('Cultural Richness Distribution')
plt.xlabel('Cultural Keywords Count')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

# 6. 길이별 품질 분석
plt.subplot(2, 3, 6)
length_bins = [0, 50, 100, 200, 500, float('inf')]
length_labels = ['<50', '50-100', '100-200', '200-500', '500+']
length_categories = pd.cut(response_lengths, bins=length_bins, labels=length_labels)
length_counts = length_categories.value_counts()
colors = ['red', 'orange', 'yellow', 'lightgreen', 'green']
length_counts.plot(kind='bar', color=colors[:len(length_counts)], alpha=0.7)
plt.title('Response Length Categories')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ 시각화 완료")

In [ ]:
# 샘플 상세 분석
print("📝 실제 데이터 샘플 분석")
print("=" * 35)

print("📋 처음 3개 샘플 상세:")
for i, (_, row) in enumerate(df.head(3).iterrows()):
    print(f"\n📖 샘플 {i+1}:")
    print(f"   ID: {row['id']}")
    print(f"   프롬프트 ({len(row['prompt'])}글자): {row['prompt']}")
    print(f"   응답 ({len(row['response'])}글자): {row['response']}")
    
    if 'metadata' in row and isinstance(row['metadata'], dict):
        metadata = row['metadata']
        print(f"   메타데이터:")
        for key, value in metadata.items():
            if key != 'extracted_elements':  # 너무 복잡한 부분 제외
                print(f"     • {key}: {value}")
    
    print("-" * 50)

# 문제점 요약
print(f"\n🎯 실제 데이터 분석 결과:")
print(f"✅ 장점:")
print(f"   • 정확히 60개 샘플 보유")
print(f"   • 다양한 제주 설화 포함")
print(f"   • 구조화된 메타데이터")
print(f"   • 토큰 시스템 활용")

print(f"\n❌ 문제점:")
print(f"   • 응답이 너무 짧음 (평균 {response_lengths.mean():.0f}글자)")
print(f"   • 모델 훈련에 부적합한 길이")
print(f"   • 스토리 내용 부족")

print(f"\n💡 개선 방안:")
print(f"   1. 응답 내용을 5-10배 확장")
print(f"   2. 스토리 배경, 등장인물, 교훈 등 상세 정보 추가")
print(f"   3. 제주도 문화적 맥락 강화")
print(f"   4. 목표: 평균 500+ 글자 응답")

print(f"\n🚀 다음 단계:")
print(f"   1. 현재 데이터를 기반으로 내용 확장 스크립트 작성")
print(f"   2. 각 샘플의 응답을 풍부한 스토리로 발전")
print(f"   3. 확장된 데이터셋으로 모델 훈련 준비")

print("\n" + "=" * 60)
print("🎉 실제 제주 설화 데이터 분석 완료! 🎉")
print(f"총 {len(df)}개 샘플, 평균 {response_lengths.mean():.0f}글자 - 확장 필요!")
print("=" * 60)